# Enron email corpus — exploratory analysis

Works on the CMU release (`enron_mail_20150507.tar.gz`, ~150 users, ~517k messages in `maildir/<user>/<folder>/<n>.`).

Pipeline: **extract tarball → parse every message into a DataFrame → cache as parquet → analyze**.
Parsing the full corpus takes a few minutes; set `MAX_FILES` to iterate quickly on a sample.

In [ ]:
from pathlib import Path
import email, email.policy, email.utils, tarfile, re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)

TARBALL = Path.home() / "Downloads" / "enron_mail_20150507.tar.gz"
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"           # tarball extracts to RAW_DIR / "maildir"
MAILDIR = RAW_DIR / "maildir"
CACHE = DATA_DIR / "enron_emails.parquet"
MAX_FILES = None                      # e.g. 20_000 for a quick sample; None = everything

## 1. Extract

In [ ]:
if not MAILDIR.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {TARBALL} ...")
    with tarfile.open(TARBALL, "r:gz") as tf:
        tf.extractall(RAW_DIR, filter="data")
users = sorted(p.name for p in MAILDIR.iterdir() if p.is_dir())
print(f"{len(users)} mailboxes, e.g. {users[:5]}")

## 2. Parse messages into a DataFrame

In [ ]:
def split_addrs(value):
    if not value:
        return []
    return [a.lower() for _, a in email.utils.getaddresses([value]) if a]

def parse_message(path):
    with open(path, "rb") as f:
        msg = email.message_from_binary_file(f, policy=email.policy.compat32)
    try:
        date = pd.Timestamp(email.utils.parsedate_to_datetime(msg["Date"]))
        date = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
    except Exception:
        date = pd.NaT
    body = msg.get_payload(decode=True) if not msg.is_multipart() else b""
    body = (body or b"").decode("latin-1", errors="replace")
    rel = path.relative_to(MAILDIR).parts
    return {
        "message_id": msg["Message-ID"],
        "date": date,
        "from": (split_addrs(msg["From"]) or [None])[0],
        "to": split_addrs(msg["To"]),
        "cc": split_addrs(msg["Cc"]),
        "bcc": split_addrs(msg["Bcc"]),
        "subject": msg["Subject"] or "",
        "x_from": msg["X-From"],
        "user": rel[0],
        "folder": "/".join(rel[1:-1]),
        "body": body,
        "path": str(path),
    }

def load_corpus(max_files=None):
    rows = []
    for i, path in enumerate(p for p in MAILDIR.rglob("*") if p.is_file()):
        if max_files and i >= max_files:
            break
        rows.append(parse_message(path))
        if i % 50_000 == 0:
            print(f"  parsed {i:,}")
    return pd.DataFrame(rows)

if CACHE.exists() and MAX_FILES is None:
    df = pd.read_parquet(CACHE)
else:
    df = load_corpus(MAX_FILES)
    if MAX_FILES is None:
        df.to_parquet(CACHE)

df["n_recipients"] = df["to"].str.len() + df["cc"].str.len() + df["bcc"].str.len()
df["body_len"] = df["body"].str.len()
print(f"{len(df):,} messages")
df.head()

## 3. Deduplicate

The same message often appears in several folders (`sent`, `sent_items`, `_sent_mail`, `all_documents`, …) and in several users' mailboxes.
Deduplicating on (sender, date, subject, body) gives the set of distinct emails.

In [ ]:
emails = df.drop_duplicates(subset=["from", "date", "subject", "body"]).copy()
print(f"{len(df):,} files -> {len(emails):,} distinct messages ({1 - len(emails)/len(df):.0%} duplicates)")

## 4. Volume over time

In [ ]:
# A handful of messages carry bogus dates (1979, 2044, ...); keep the real window.
emails = emails[(emails["date"] >= "1998-01-01") & (emails["date"] < "2003-01-01")]

monthly = emails.set_index("date").resample("MS").size()
ax = monthly.plot(figsize=(11, 3.5), color="#4C72B0")
ax.set(title="Distinct emails per month", xlabel="", ylabel="emails")
for d, label in [("2001-08-14", "Skilling resigns"), ("2001-10-16", "Q3 loss"), ("2001-12-02", "Bankruptcy")]:
    ax.axvline(pd.Timestamp(d, tz="UTC"), color="gray", ls="--", lw=0.8)
    ax.text(pd.Timestamp(d, tz="UTC"), ax.get_ylim()[1] * 0.95, label, rotation=90, va="top", ha="right", fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
local = emails["date"].dt.tz_convert("US/Central")  # Enron HQ was in Houston
heat = pd.crosstab(local.dt.day_of_week, local.dt.hour).reindex(index=range(7), columns=range(24), fill_value=0)
heat.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(11, 3))
im = ax.imshow(heat, aspect="auto", cmap="Blues")
ax.set_xticks(range(24)); ax.set_yticks(range(7)); ax.set_yticklabels(heat.index)
ax.set(title="When were emails sent? (Houston time)", xlabel="hour of day")
fig.colorbar(im, ax=ax, label="emails")
plt.show()

## 5. Who sends and receives the most

In [ ]:
top_senders = emails["from"].value_counts().head(20)
top_recipients = emails.explode("to")["to"].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, s, title in [(axes[0], top_senders, "Top senders"), (axes[1], top_recipients, "Top recipients (To:)")]:
    s.sort_values().plot.barh(ax=ax, color="#4C72B0")
    ax.set(title=title, xlabel="emails", ylabel="")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
is_internal = lambda a: isinstance(a, str) and a.endswith("@enron.com")
print(f"Sent from @enron.com: {emails['from'].map(is_internal).mean():.1%}")
domains = emails["from"].dropna().str.split("@").str[-1].value_counts().head(15)
domains

## 6. Mailboxes and folders

In [ ]:
per_user = df.groupby("user").size().sort_values(ascending=False)
ax = per_user.head(25).sort_values().plot.barh(figsize=(8, 7), color="#4C72B0")
ax.set(title="Largest mailboxes (files, incl. duplicates)", xlabel="files", ylabel="")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

df["folder"].str.split("/").str[0].value_counts().head(20)

## 7. Communication network (who emails whom)

In [ ]:
edges = (
    emails.explode("to")
    .dropna(subset=["from", "to"])
    .query("`from` != to")
    .groupby(["from", "to"]).size()
    .rename("n").sort_values(ascending=False)
    .reset_index()
)
print(f"{len(edges):,} distinct sender->recipient pairs")
edges.head(25)

## 8. Message content

In [ ]:
print(emails["body_len"].describe().round(0))
ax = emails["body_len"].clip(upper=10_000).plot.hist(bins=80, figsize=(10, 3), color="#4C72B0")
ax.set(title="Body length (chars, clipped at 10k)", xlabel="characters")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

ax = emails["n_recipients"].clip(upper=50).plot.hist(bins=50, figsize=(10, 3), color="#4C72B0")
ax.set(title="Recipients per email (clipped at 50)", xlabel="to + cc + bcc")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
STOP = set("""re fw fwd the a an and or of to for in on at is with from by your you our we this that
it be are as - & : / meeting update""".split())
words = Counter(
    w for s in emails["subject"].str.lower()
    for w in re.findall(r"[a-z][a-z']+", s) if w not in STOP and len(w) > 2
)
pd.Series(dict(words.most_common(30))).to_frame("count").T

In [ ]:
def search(term, n=10):
    """Case-insensitive search over subject + body."""
    hit = emails["subject"].str.contains(term, case=False, regex=False) | emails["body"].str.contains(term, case=False, regex=False)
    print(f"{hit.sum():,} emails mention {term!r}")
    return emails.loc[hit, ["date", "from", "subject"]].sort_values("date").head(n)

search("raptor")